# 🎯 Açık Kaynak Modeller — Base & Varyant Test Notebook (FINAL)

Bu notebook, 4 açık kaynak modeli **Base (489 soru)** veya **Varyant (258 soru)** veri seti üzerinde test eder.

**Test edilecek modeller:**
1. Qwen 2.5 7B Instruct
2. Trendyol LLM 7B Chat
3. Llama 3.1 8B Instruct
4. Gemma 2 9B IT

**Skorlama:** `core/faithfulness_utils.py` dosyası **hiç değiştirilmeden** kullanılır.

**Donanım:** Colab Pro (A100 önerilir, V100/L4 da çalışır)

---

## 📋 Kullanım Sırası

1. **Bölüm A** (Setup) - **Her runtime başında zorunlu**
2. **Bölüm B** (Yardımcı fonksiyonlar) - **Her runtime başında zorunlu**
3. Hangi model için skorlama yapacaksan o bölümü (C/D/E/F) çalıştır
4. Her model bitince GPU temizlenir, sonraki modele geçilebilir

## ⚙️ Base / Varyant Geçişi

**Bölüm A.4'te `DATASET_MODE` değişkenini değiştir:**
- `"base"` → 489 soruluk Base_Questions
- `"varyant"` → 258 soruluk Varyant veri seti

Diğer her şey otomatik yönetilir (CSV adı, checkpoint adı, sonuç adı).

---
## 🔧 BÖLÜM A — Setup (Her zaman çalıştır)
---

In [ ]:
# ============================================================
# A.1 — Kütüphaneleri kur
# ============================================================

!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers
!pip install -q tqdm scipy

print("✓ Kütüphaneler kuruldu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.8 MB/s eta 0:00:00
✓ Kütüphaneler kuruldu


In [ ]:
# ============================================================
# A.2 — Drive bağla, path'i hazırla, cache temizle
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import sys, os, shutil, gc, json, time

PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
sys.path.insert(0, PROJECT_FOLDER)

# Eski cache'i temizle (yeni import'u garanti et)
pycache_path = f"{PROJECT_FOLDER}/core/__pycache__"
if os.path.exists(pycache_path):
    shutil.rmtree(pycache_path)
    print("✓ __pycache__ temizlendi")

utils_path = f"{PROJECT_FOLDER}/core/faithfulness_utils.py"
assert os.path.exists(utils_path), f"faithfulness_utils.py bulunamadı: {utils_path}"
print(f"✓ faithfulness_utils.py bulundu: {utils_path}")

# Tüm fonksiyonları import et (template notebook'u ile birebir aynı yöntem)
from core.faithfulness_utils import *

print("✓ faithfulness_utils fonksiyonları import edildi")

Mounted at /content/drive
✓ __pycache__ temizlendi
✓ faithfulness_utils.py bulundu: /content/drive/MyDrive/BitirmeProjesi/core/faithfulness_utils.py
✓ faithfulness_utils fonksiyonları import edildi


In [ ]:
# ============================================================
# A.3 — Transformers, GPU kontrolü, HuggingFace token
# ============================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from google.colab import userdata

# GPU kontrolü
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    raise RuntimeError("GPU yok! Runtime → Change runtime type → GPU seç")

# HuggingFace login (Colab Secrets'a HF_TOKEN ekli olmalı)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print("✓ HuggingFace login başarılı")
except Exception as e:
    print(f"⚠ HF token sorunu: {e}")
    print("  Colab Secrets'a HF_TOKEN ekle: huggingface.co/settings/tokens")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB (39.5 GB VRAM)
✓ HuggingFace login başarılı


In [ ]:
# ============================================================
# A.4 — Veri seti seçimi (BASE veya VARYANT)
# ============================================================
# Buradan kontrol et: "base" mi "varyant" mı?

DATASET_MODE = "base"   # <<< "base" veya "varyant" yap

import pandas as pd
from datetime import datetime

# Path'ler
CHECKPOINT_PATH = f"{PROJECT_FOLDER}/checkpoints/"
RESULTS_PATH = f"{PROJECT_FOLDER}/results/"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

if DATASET_MODE == "base":
    DATA_PATH = f"{PROJECT_FOLDER}/Sorular e-tablo - Base_Questions 489.csv"
    DATASET_TAG = "base"
    EXPECTED_ROWS = 489
elif DATASET_MODE == "varyant":
    # Bu CSV adını kendi varyant dosyanın adıyla değiştir
    DATA_PATH = f"{PROJECT_FOLDER}/Sorular e-tablo - Varyant_Questions 258.csv"
    DATASET_TAG = "varyant"
    EXPECTED_ROWS = 258
else:
    raise ValueError(f"DATASET_MODE 'base' veya 'varyant' olmalı, geldi: {DATASET_MODE}")

# Config'i utils'e set et (faithfulness için ağırlıklar)
set_config(
    vector_w=0.4,
    nli_w=0.6,
    llm_model="local-open-source",   # sadece etiket, generation'da kullanılmıyor
    temp=0,
    max_tok=2048,
    checkpoint_path=CHECKPOINT_PATH
)

# Tarih etiketi
TODAY = datetime.now().strftime('%Y%m%d')

# Veriyi yükle ve preprocess et (gold_evidence_list ve evidence_count kolonlarını oluşturur)
df = load_and_preprocess_data(DATA_PATH)

print(f"\n✓ Veri seti yüklendi: {len(df)} satır")
print(f"  Mode: {DATASET_MODE} (beklenen: {EXPECTED_ROWS})")
print(f"  Kolonlar: {list(df.columns)}")
print(f"  Tarih: {TODAY}")
print(f"  Sonuçlar: {RESULTS_PATH}")

✓ Veri yüklendi: 489 satır
  ⚠️ 1 satır eksik veri nedeniyle çıkarıldı
  Evidence parça sayıları: min=1, max=7, ortalama=2.8

✓ Veri seti yüklendi: 488 satır
  Mode: base (beklenen: 489)
  Kolonlar: ['question_id', 'category', 'difficulty', 'context', 'question', 'gold_evidence', 'gold_answer', 'gold_evidence_list', 'evidence_count']
  Tarih: 20260514
  Sonuçlar: /content/drive/MyDrive/BitirmeProjesi/results/


In [ ]:
# ============================================================
# A.5 — Embedding ve NLI modellerini yükle, set_models ile kaydet
# ============================================================
# KRİTİK: NLI pipeline'ı "zero-shot-classification" olarak yüklenmeli
# (text-classification DEĞİL — faithfulness_utils zero-shot bekliyor)

EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
NLI_MODEL = "joeddav/xlm-roberta-large-xnli"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("► Embedding model yükleniyor...")
emb_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f"  ✓ {EMBEDDING_MODEL}")

print("► NLI pipeline yükleniyor (zero-shot-classification)...")
nli_pipe = pipeline(
    "zero-shot-classification",
    model=NLI_MODEL,
    device=0 if device == "cuda" else -1
)
print(f"  ✓ {NLI_MODEL}")

# Global olarak faithfulness_utils'e tanıt
set_models(emb_model, nli_pipe, None, None, None)

# Hızlı sanity check
test_sem = calculate_semantic_similarity("merhaba dünya", "selam dünya")
test_nli = calculate_nli_score("Bugün hava güzel", "Hava bugün güzel")
print(f"\n  Sanity check:")
print(f"    semantic_similarity test = {test_sem:.4f}")
print(f"    nli_score test = {test_nli['nli_score']:.4f} (label={test_nli['label']})")
print("\n✓ Skorlama altyapısı hazır")

► Embedding model yükleniyor...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

  ✓ joeddav/xlm-roberta-large-xnli

  Sanity check:
    semantic_similarity test = 0.9570
    nli_score test = 0.3886 (label=contradiction)

✓ Skorlama altyapısı hazır


---
## 🔧 BÖLÜM B — Yardımcı Fonksiyonlar (Her zaman çalıştır)
---

In [ ]:
# ============================================================
# B.1 — Prompt template (faithfulness_utils.SYSTEM_PROMPT ile uyumlu)
# ============================================================

# faithfulness_utils.py içindeki SYSTEM_PROMPT'u kullan, böylece base testleriyle aynı format
# (zaten import * ile geldi)
OPEN_SOURCE_SYSTEM_PROMPT = SYSTEM_PROMPT

def build_user_prompt(context: str, question: str) -> str:
    return f"""BAĞLAM:
{context}

SORU:
{question}

Lütfen yukarıdaki kurallara uygun olarak SADECE geçerli JSON formatında cevap ver."""

print("✓ Prompt template hazır (faithfulness_utils.SYSTEM_PROMPT kullanılıyor)")
print(f"  SYSTEM_PROMPT uzunluğu: {len(SYSTEM_PROMPT)} karakter")

✓ Prompt template hazır (faithfulness_utils.SYSTEM_PROMPT kullanılıyor)
  SYSTEM_PROMPT uzunluğu: 659 karakter


In [ ]:
# ============================================================
# B.2 — Model yükleme ve generation fonksiyonları
# ============================================================

def load_model_4bit(model_id: str):
    """Modeli 4-bit quantization ile yükler (VRAM tasarrufu)."""
    print(f"► Model yükleniyor: {model_id}")
    print("  (İlk indirme uzun sürebilir, sonraki çalıştırmalar cache'den hızlı)")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"✓ Model yüklendi: {model_id}")
    return model, tokenizer


def _supports_system_role(tokenizer) -> bool:
    """Tokenizer'ın chat template'i system role'unü destekliyor mu test et.
    Gemma gibi modeller system role'unu doğrudan kabul etmez."""
    try:
        tokenizer.apply_chat_template(
            [{"role": "system", "content": "test"}, {"role": "user", "content": "test"}],
            tokenize=False,
            add_generation_prompt=True,
        )
        return True
    except Exception:
        return False


def generate_response(model, tokenizer, context: str, question: str, max_new_tokens: int = 768) -> str:
    """Tek bir soru için modelden cevap al."""
    user_prompt = build_user_prompt(context, question)

    if _supports_system_role(tokenizer):
        messages = [
            {"role": "system", "content": OPEN_SOURCE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
    else:
        # Gemma vb. için system'i user'a katlayıp tek mesaj olarak gönder
        combined = f"{OPEN_SOURCE_SYSTEM_PROMPT}\n\n{user_prompt}"
        messages = [{"role": "user", "content": combined}]

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                       # temperature=0 davranışı
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Sadece üretilen kısmı al (input'u çıkar)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    return response


def cleanup_model(model, tokenizer):
    """Modeli bellekten temizle."""
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("✓ GPU belleği temizlendi")


print("✓ Model yükleme/generation fonksiyonları tanımlandı")

✓ Model yükleme/generation fonksiyonları tanımlandı


In [ ]:
# ============================================================
# B.3 — Generation: checkpoint'li tüm veri seti üzerinde model çalıştır
# ============================================================

from tqdm.notebook import tqdm

def run_generation(model, tokenizer, df, model_short_name: str, dataset_tag: str):
    """
    Modeli tüm veri seti üzerinde çalıştırır.
    Checkpoint dosyası: generation_{model}_{dataset}_{tarih}.json
    Her 5 soruda bir kaydedilir.
    """
    checkpoint_file = f"{CHECKPOINT_PATH}generation_{model_short_name}_{dataset_tag}_{TODAY}.json"

    results = []
    processed_ids = set()
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
        processed_ids = {r['question_id'] for r in results}
        print(f"✓ Checkpoint bulundu: {len(processed_ids)} soru zaten işlenmiş")

    to_process = df[~df['question_id'].isin(processed_ids)]

    if len(to_process) == 0:
        print("✓ Tüm sorular zaten işlenmiş — checkpoint'ten devam ediliyor")
    else:
        print(f"► {len(to_process)} soru işlenecek (model: {model_short_name})")

        for idx, row in tqdm(to_process.iterrows(), total=len(to_process), desc=f"Gen [{model_short_name}]"):
            context = str(row['context'])
            question = str(row['question'])

            try:
                raw = generate_response(model, tokenizer, context, question)
                parsed = extract_json_from_response(raw)

                if parsed is None or not isinstance(parsed, dict):
                    results.append({
                        'question_id': row['question_id'],
                        'model_raw_response': raw,
                        'model_reasoning': None,
                        'model_answer': None,
                        'model_confidence': None,
                        'error': 'JSON_PARSE_ERROR',
                    })
                else:
                    results.append({
                        'question_id': row['question_id'],
                        'model_raw_response': raw,
                        'model_reasoning': parsed.get('reasoning'),
                        'model_answer': parsed.get('answer'),
                        'model_confidence': parsed.get('confidence'),
                        'error': None,
                    })
            except Exception as e:
                results.append({
                    'question_id': row['question_id'],
                    'model_raw_response': '',
                    'model_reasoning': None,
                    'model_answer': None,
                    'model_confidence': None,
                    'error': str(e),
                })

            # Her 5 satırda bir kaydet (data loss önlemi)
            if len(results) % 5 == 0:
                with open(checkpoint_file, 'w', encoding='utf-8') as f:
                    json.dump(results, f, ensure_ascii=False, indent=2)

        # Son kayıt
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"✓ Checkpoint kaydedildi: {checkpoint_file}")

    # df ile birleştir
    results_df = pd.DataFrame(results)
    merged = df.merge(results_df, on='question_id', how='left')

    success_count = merged['model_answer'].notna().sum()
    error_count = merged['error'].notna().sum()
    print(f"\n✓ Generation özeti:")
    print(f"   Başarılı: {success_count}/{len(merged)}")
    if error_count > 0:
        print(f"   Hatalı (JSON parse vb.): {error_count}")

    return merged

print("✓ run_generation fonksiyonu tanımlandı")

✓ run_generation fonksiyonu tanımlandı


In [ ]:
# ============================================================
# B.4 — Skorlama: faithfulness_utils'in HAZIR fonksiyonlarını kullanır
# ============================================================
# Hiçbir fonksiyon imzasını yeniden tanımlamıyoruz, doğrudan utils'inkini çağırıyoruz.
# Bu da base testleriyle metodolojik karşılaştırılabilirliği garantiler.

def run_scoring(merged_df: pd.DataFrame, model_short_name: str, dataset_tag: str) -> pd.DataFrame:
    """
    Generation sonuçlarını skorlar.
    Adım 1: run_evaluation -> faithfulness, exact_match, epistemic metrics
    Adım 2: soft accuracy ekle (örnek bazında)
    """
    # Sadece geçerli (parse edilmiş) satırları skorla
    valid = merged_df[
        merged_df['error'].isna() &
        merged_df['model_reasoning'].notna() &
        (merged_df['model_reasoning'].astype(str).str.len() > 0)
    ].copy().reset_index(drop=True)

    print(f"► Skorlanacak satır sayısı: {len(valid)} / {len(merged_df)}")
    if len(valid) == 0:
        print("⚠ Hiç geçerli satır yok, skorlama yapılamaz")
        return merged_df

    # Adım 1: Faithfulness + accuracy + epistemic (utils'in tek fonksiyonu yapıyor)
    df_evaluated = run_evaluation(valid)

    # Adım 2: Soft accuracy ekle (örnek bazında)
    print("\n► Soft accuracy hesaplanıyor...")
    soft_rows = []
    for _, row in tqdm(df_evaluated.iterrows(), total=len(df_evaluated), desc="Soft Acc"):
        try:
            r = calculate_soft_accuracy(row['model_answer'], row['gold_answer'])
        except Exception as e:
            r = {
                'soft_match': False,
                'accuracy_score': 0,
                'match_type': f'error: {e}',
                'semantic_sim': 0.0,
                'nli_entailment': 0.0,
            }
        soft_rows.append({
            'question_id': row['question_id'],
            'soft_match': r['soft_match'],
            'soft_accuracy_score': r['accuracy_score'],
            'match_type': r['match_type'],
            'answer_semantic_sim': r['semantic_sim'],
            'answer_nli_entailment': r['nli_entailment'],
        })
    soft_df = pd.DataFrame(soft_rows)
    df_evaluated = df_evaluated.merge(soft_df, on='question_id', how='left')

    # Sonucu kaydet
    output_path = f"{RESULTS_PATH}evaluation_acik_{model_short_name}_{dataset_tag}_{TODAY}.csv"
    df_evaluated.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ Skorlama tamamlandı, kaydedildi: {output_path}")

    return df_evaluated


def print_summary(df_evaluated: pd.DataFrame, model_display_name: str):
    """Sonuç özetini bas. (Hem exact hem soft accuracy gösterilir.)"""
    print("\n" + "=" * 70)
    print(f"📊 ÖZET — {model_display_name}")
    print("=" * 70)
    print(f"Toplam soru: {len(df_evaluated)}")

    if 'accuracy_score' in df_evaluated.columns:
        exact_rate = df_evaluated['accuracy_score'].mean() * 100
        print(f"\nExact Accuracy: {exact_rate:.1f}%  ({df_evaluated['accuracy_score'].sum()}/{len(df_evaluated)})")

    if 'soft_accuracy_score' in df_evaluated.columns:
        soft_rate = df_evaluated['soft_accuracy_score'].mean() * 100
        print(f"Soft Accuracy:  {soft_rate:.1f}%  ({df_evaluated['soft_accuracy_score'].sum()}/{len(df_evaluated)})")

    if 'faithfulness_score' in df_evaluated.columns:
        print(f"\nOrtalama semantic_similarity: {df_evaluated['semantic_similarity'].mean():.3f}")
        print(f"Ortalama nli_score:           {df_evaluated['nli_score'].mean():.3f}")
        print(f"Ortalama faithfulness_score:  {df_evaluated['faithfulness_score'].mean():.3f}")
        print(f"Ortalama evidence_recall:     {df_evaluated['evidence_recall'].mean():.3f}")

    if 'confidence_normalized' in df_evaluated.columns:
        print(f"\nOrtalama confidence: {df_evaluated['confidence_normalized'].mean()*100:.1f}%")
        print(f"Ortalama epistemic_gap: {df_evaluated['epistemic_gap'].mean():.3f}")

        print("\nHallucination risk dağılımı:")
        for risk, cnt in df_evaluated['hallucination_risk'].value_counts().items():
            print(f"   {risk}: {cnt} ({cnt/len(df_evaluated)*100:.1f}%)")

        print("\nCalibration tipi dağılımı:")
        for cal, cnt in df_evaluated['calibration_type'].value_counts().items():
            print(f"   {cal}: {cnt} ({cnt/len(df_evaluated)*100:.1f}%)")

    print("=" * 70)


print("✓ run_scoring ve print_summary tanımlandı")

✓ run_scoring ve print_summary tanımlandı


---
## 🔧 BÖLÜM C — Qwen 2.5 7B Instruct

**Model:** Qwen/Qwen2.5-7B-Instruct  
**Tahmini süre:** ~1-1.5 saat (489 soru) / ~30-45 dk (258 varyant)  
**GPU bellek:** ~6-7 GB (4-bit)
---

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_SHORT_NAME = "qwen25_7b"
MODEL_DISPLAY_NAME = "Qwen 2.5 7B Instruct"

# 1) Modeli yükle
model, tokenizer = load_model_4bit(MODEL_ID)

# 2) Generation
merged = run_generation(model, tokenizer, df, MODEL_SHORT_NAME, DATASET_TAG)

# 3) Generation bittikten sonra LLM'i bellekten at (skorlama için yer aç)
cleanup_model(model, tokenizer)

# 4) Skorlama (sentence-transformers + NLI hâlâ yüklü)
df_evaluated = run_scoring(merged, MODEL_SHORT_NAME, DATASET_TAG)

# 5) Özet
print_summary(df_evaluated, MODEL_DISPLAY_NAME)

NameError: name 'load_model_4bit' is not defined

---
## 🔧 BÖLÜM D — Trendyol LLM 7B Chat

**Model:** Trendyol/Trendyol-LLM-7b-chat-v1.0  
**Özellik:** Türkçe-spesifik fine-tune
---

In [ ]:
MODEL_ID = "Trendyol/Trendyol-LLM-7b-chat-v1.0"
MODEL_SHORT_NAME = "trendyol_7b"
MODEL_DISPLAY_NAME = "Trendyol LLM 7B"

model, tokenizer = load_model_4bit(MODEL_ID)
merged = run_generation(model, tokenizer, df, MODEL_SHORT_NAME, DATASET_TAG)
cleanup_model(model, tokenizer)

df_evaluated = run_scoring(merged, MODEL_SHORT_NAME, DATASET_TAG)
print_summary(df_evaluated, MODEL_DISPLAY_NAME)

► Model yükleniyor: Trendyol/Trendyol-LLM-7b-chat-v1.0
  (İlk indirme uzun sürebilir, sonraki çalıştırmalar cache'den hızlı)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✓ Model yüklendi: Trendyol/Trendyol-LLM-7b-chat-v1.0
✓ Checkpoint bulundu: 300 soru zaten işlenmiş
► 188 soru işlenecek (model: trendyol_7b)


Gen [trendyol_7b]:   0%|          | 0/188 [00:00<?, ?it/s]

✓ Checkpoint kaydedildi: /content/drive/MyDrive/BitirmeProjesi/checkpoints/generation_trendyol_7b_base_20260514.json

✓ Generation özeti:
   Başarılı: 0/488
   Hatalı (JSON parse vb.): 488
✓ GPU belleği temizlendi
► Skorlanacak satır sayısı: 0 / 488
⚠ Hiç geçerli satır yok, skorlama yapılamaz

📊 ÖZET — Trendyol LLM 7B
Toplam soru: 488


---
## 🔧 BÖLÜM E — Llama 3.1 8B Instruct

**Model:** meta-llama/Llama-3.1-8B-Instruct  
**Not:** HuggingFace'de erişim onayı gerekiyor (HF_TOKEN ile)
---

In [ ]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_SHORT_NAME = "llama31_8b"
MODEL_DISPLAY_NAME = "Llama 3.1 8B Instruct"

model, tokenizer = load_model_4bit(MODEL_ID)
merged = run_generation(model, tokenizer, df, MODEL_SHORT_NAME, DATASET_TAG)
cleanup_model(model, tokenizer)

df_evaluated = run_scoring(merged, MODEL_SHORT_NAME, DATASET_TAG)
print_summary(df_evaluated, MODEL_DISPLAY_NAME)

► Model yükleniyor: meta-llama/Llama-3.1-8B-Instruct
  (İlk indirme uzun sürebilir, sonraki çalıştırmalar cache'den hızlı)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Evaluating:   0%|          | 0/303 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



✓ Evaluation tamamlandı!

► Soft accuracy hesaplanıyor...


Soft Acc:   0%|          | 0/303 [00:00<?, ?it/s]


✓ Skorlama tamamlandı, kaydedildi: /content/drive/MyDrive/BitirmeProjesi/results/evaluation_acik_llama31_8b_base_20260514.csv

📊 ÖZET — Llama 3.1 8B Instruct
Toplam soru: 303

Exact Accuracy: 29.0%  (88/303)
Soft Accuracy:  40.3%  (122/303)

Ortalama semantic_similarity: 0.875
Ortalama nli_score:           0.510
Ortalama faithfulness_score:  0.656
Ortalama evidence_recall:     1.000

Ortalama confidence: 97.4%
Ortalama epistemic_gap: 0.321

Hallucination risk dağılımı:
   ORTA: 198 (65.3%)
   DÜŞÜK: 89 (29.4%)
   YÜKSEK: 16 (5.3%)

Calibration tipi dağılımı:
   Overconfident: 282 (93.1%)
   İyi Kalibre: 20 (6.6%)
   Underconfident: 1 (0.3%)


In [ ]:
import json

checkpoint_file = "/content/drive/MyDrive/BitirmeProjesi/checkpoints/generation_llama31_8b_base_20260514.json"

with open(checkpoint_file, 'r', encoding='utf-8') as f:
    results = json.load(f)

# Sadece hatalıları al
errors = [r for r in results if r.get('error') is not None]
print(f"Toplam: {len(results)}, Hatalı: {len(errors)}")
print(f"\nİlk 3 hatalı yanıt:\n")
for i, r in enumerate(errors[:3]):
    print(f"--- Hata {i+1} (question_id={r['question_id']}) ---")
    print(f"Error: {r.get('error')}")
    print(f"Raw response (ilk 800 karakter):")
    raw = r.get('model_raw_response', '')
    print(raw[:800])
    print("\n" + "="*60 + "\n")

Toplam: 488, Hatalı: 185

İlk 3 hatalı yanıt:

--- Hata 1 (question_id=NR-1-001) ---
Error: JSON_PARSE_ERROR
Raw response (ilk 800 karakter):
{
    "reasoning": "Adım 1: C sayısının yüzler basamağını 4 artırarak elde edilen sonucun D sayısından farklı olması gerektiği anlaşılıyor. Bu, C sayısının yüzler basamağını 4 artırarak elde edilen sonucun D sayısından 4 daha büyük olması gerektiği anlamına gelir.\nAdım 2: C sayısının yüzler basamağını 4 artırarak elde edilen sonucun D sayısından 4 daha büyük olması gerektiği anlaşılıyor. Bu, C sayısının yüzler basamağını 4 artırarak elde edilen sonucun D sayısından 4 daha büyük olması gerektiği anlamına gelir.\nAdım 3: C sayısının yüzler basamağını 4 artırarak elde edilen sonucun D sayısından 4 daha büyük olması gerektiği anlaşılıyor. Bu, C sayısının yüzler basamağını 4 artırarak elde edilen sonucun D sayısından 4 daha büyük olması gerektiği anlamına gelir.\nAdım 4: C sayısının yüzler bas


--- Hata 2 (question_id=NR-2-003) ---
Error: JSON_PARSE

---
## 🔧 BÖLÜM F — Gemma 2 9B IT

**Model:** google/gemma-2-9b-it  
**GPU bellek:** ~8-9 GB (4-bit)  
**Not:** Gemma system role'unu desteklemez, fallback otomatik devreye girer.
---

In [ ]:
MODEL_ID = "google/gemma-2-9b-it"
MODEL_SHORT_NAME = "gemma2_9b"
MODEL_DISPLAY_NAME = "Gemma 2 9B IT"

model, tokenizer = load_model_4bit(MODEL_ID)
merged = run_generation(model, tokenizer, df, MODEL_SHORT_NAME, DATASET_TAG)
cleanup_model(model, tokenizer)

df_evaluated = run_scoring(merged, MODEL_SHORT_NAME, DATASET_TAG)
print_summary(df_evaluated, MODEL_DISPLAY_NAME)

► Model yükleniyor: google/gemma-2-9b-it
  (İlk indirme uzun sürebilir, sonraki çalıştırmalar cache'den hızlı)


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

✓ Model yüklendi: google/gemma-2-9b-it
► 488 soru işlenecek (model: gemma2_9b)


Gen [gemma2_9b]:   0%|          | 0/488 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## 🔧 BÖLÜM G — (Opsiyonel) Sonradan Skorlama / Yeniden Skorlama

Eğer generation tamamlandıktan sonra runtime kapanırsa (ya da skorlamayı sonra yapmak istersen), bu bölüm checkpoint dosyasından okuyup sadece skorlama yapar. **Model yüklemez, sadece sentence-transformers + NLI yeterli.**

Önce Bölüm A (1-5) ve Bölüm B'yi çalıştırmış olman gerekir.
---

In [ ]:
# Hangi modeli yeniden skorlamak istiyorsun, onu seç
RESCORE_MODEL_SHORT = "qwen25_7b"   # ör: "qwen25_7b", "trendyol_7b", "llama31_8b", "gemma2_9b"
RESCORE_DISPLAY = "Qwen 2.5 7B Instruct"

# Eski tarihli bir checkpoint'i de okuyabilirsin, gerekirse aşağıyı düzenle
rescore_checkpoint = f"{CHECKPOINT_PATH}generation_{RESCORE_MODEL_SHORT}_{DATASET_TAG}_{TODAY}.json"

if not os.path.exists(rescore_checkpoint):
    # Aynı modelin başka tarihli checkpoint'ini ara
    import glob
    candidates = glob.glob(f"{CHECKPOINT_PATH}generation_{RESCORE_MODEL_SHORT}_{DATASET_TAG}_*.json")
    if candidates:
        rescore_checkpoint = sorted(candidates)[-1]   # en yenisi
        print(f"⚠ Bugünkü checkpoint yok, en yenisi kullanılıyor: {rescore_checkpoint}")
    else:
        raise FileNotFoundError(f"Checkpoint bulunamadı: {RESCORE_MODEL_SHORT} / {DATASET_TAG}")

with open(rescore_checkpoint, 'r', encoding='utf-8') as f:
    gen_results = json.load(f)

gen_df = pd.DataFrame(gen_results)
merged = df.merge(gen_df, on='question_id', how='left')
print(f"✓ Generation checkpoint yüklendi: {len(merged)} satır")

df_evaluated = run_scoring(merged, RESCORE_MODEL_SHORT, DATASET_TAG)
print_summary(df_evaluated, RESCORE_DISPLAY)